<a href="https://colab.research.google.com/github/swarnamalyamohan/AI-agent-based-log-aware-incident-analysis/blob/main/notebooks/ai_agent_log_aware_incidents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/swarnamalyamohan/AI-agent-based-log-aware-incident-analysis.git
%cd AI-agent-based-log-aware-incident-analysis
!pip -q install -r requirements.txt
!pip -q install jupyter ipykernel

Cloning into 'AI-agent-based-log-aware-incident-analysis'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 24 (delta 2), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 8.57 KiB | 8.57 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/AI-agent-based-log-aware-incident-analysis
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.5 MB/s eta 0:00:00


In [2]:
import os
import sys
from dotenv import load_dotenv
from google.colab import userdata



# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable (comment out if not using OpenAI)
if not userdata.get('OPENAI_API_KEY'):
    os.environ["OPENAI_API_KEY"] = input("Please enter your OpenAI API key: ")
else:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [3]:
from incident_rag.config import Config
from incident_rag.agent import IncidentAssistantAgent
from incident_rag.log_parser import LogParser
from incident_rag.log_analyzer import LogAnalyzer
from incident_rag.rag_pipeline import IncidentRAGPipeline
from incident_rag.triage_generator import TriageGenerator
from incident_rag.embedding_service import EmbeddingService
from incident_rag.vector_store import LocalVectorStore

print("Repo modules imported successfully")
print("Generation model:", Config.GENERATION_MODEL)
print("Embedding model:", Config.EMBEDDING_MODEL)
print("Top K:", Config.TOP_K)

Repo modules imported successfully
Generation model: gpt-4.1-mini
Embedding model: text-embedding-3-small
Top K: 5


In [4]:
incident_text = """
Payments API latency spiked after deployment.
Error rate increased from 1% to 18%.
Customers are reporting timeouts during checkout.
"""

log_text = """
2026-04-26 10:21:11 INFO service=payments-api Starting request handling
2026-04-26 10:21:12 ERROR service=payments-api Connection pool exhausted for datasource main
2026-04-26 10:21:12 ERROR service=payments-api Failed to acquire DB connection after 30000ms
2026-04-26 10:21:13 WARN service=payments-api Request timed out for /checkout
2026-04-26 10:21:14 ERROR service=payments-api SQLException: too many connections
2026-04-26 10:21:15 ERROR service=payments-api Request failed with status 500
2026-04-26 10:21:18 INFO service=payments-api Deployment version 2.8.4 activated
2026-04-26 10:21:20 WARN service=payments-api Health check latency above threshold
2026-04-26 10:21:22 ERROR service=payments-api Timeout while calling downstream order service
2026-04-26 10:21:24 ERROR service=payments-api Failed request correlationId=abc-123 status=500
"""

print("Incident and logs loaded")

Incident and logs loaded


In [5]:
parser = LogParser()
parsed_chunks = parser.parse_text(log_text)

print(f"Parsed chunk count: {len(parsed_chunks)}")
print("\nFirst parsed chunk:")
print(parsed_chunks[0] if parsed_chunks else "No chunks parsed")

Parsed chunk count: 2

First parsed chunk:
{'chunk_id': 'log_chunk_1', 'source': 'runtime_logs', 'section': 'log_window', 'text': '2026-04-26 10:21:11 INFO service=payments-api Starting request handling\n2026-04-26 10:21:12 ERROR service=payments-api Connection pool exhausted for datasource main\n2026-04-26 10:21:12 ERROR service=payments-api Failed to acquire DB connection after 30000ms\n2026-04-26 10:21:13 WARN service=payments-api Request timed out for /checkout\n2026-04-26 10:21:14 ERROR service=payments-api SQLException: too many connections\n2026-04-26 10:21:15 ERROR service=payments-api Request failed with status 500\n2026-04-26 10:21:18 INFO service=payments-api Deployment version 2.8.4 activated\n2026-04-26 10:21:20 WARN service=payments-api Health check latency above threshold\n2026-04-26 10:21:22 ERROR service=payments-api Timeout while calling downstream order service\n2026-04-26 10:21:24 ERROR service=payments-api Failed request correlationId=abc-123 status=500', 'timestam

In [6]:
agent = IncidentAssistantAgent()
agent.build_knowledge_base("incidents")
print("Historical incident knowledge base built from incidents/")

Loading incident documents...
Loaded 8 incident chunks
Generating embeddings...
Building local FAISS index...
FAISS index created with 8 vectors
Knowledge base is ready
Historical incident knowledge base built from incidents/


In [13]:
# Compare logs-only vs logs+incident-history

agent = IncidentAssistantAgent()
agent.build_knowledge_base("incidents")
agent.build_log_index_from_text(log_text)

relevant_logs = agent.retrieve_relevant_logs(incident_text)
log_analysis = agent.log_analyzer.analyze(incident_text, relevant_logs)
similar_incidents = agent.pipeline.retrieve_similar_incidents(incident_text)

logs_only_triage = agent.triage_generator.generate_with_logs(
    new_incident=incident_text,
    similar_incidents=[],
    relevant_logs=relevant_logs,
    log_analysis=log_analysis,
)

full_triage = agent.triage_generator.generate_with_logs(
    new_incident=incident_text,
    similar_incidents=similar_incidents,
    relevant_logs=relevant_logs,
    log_analysis=log_analysis,
)

print("===== LOGS ONLY =====")
print(logs_only_triage)

print("\n\n===== LOGS + INCIDENT HISTORY =====")
print(full_triage)

Loading incident documents...
Loaded 8 incident chunks
Generating embeddings...
Building local FAISS index...
FAISS index created with 8 vectors
Knowledge base is ready
FAISS index created with 2 vectors
===== LOGS ONLY =====
1. Likely Root Cause  
The deployment of payments-api version 2.8.4 introduced a regression that caused exhaustion of the database connection pool for the main datasource. This led to failures in acquiring DB connections, resulting in SQL exceptions ("too many connections"), increased request latency, timeouts on /checkout requests, and elevated HTTP 500 error rates. The downstream order service timeouts appear to be a secondary effect caused by cascading delays or resource contention.

2. Evidence from Logs  
- At 10:21:12, logs show "Connection pool exhausted for datasource main" and "Failed to acquire DB connection after 30000ms."  
- SQLException errors citing "too many connections" immediately follow.  
- Requests to /checkout time out and return HTTP 500 err

In [7]:
result = agent.run(
    new_incident=incident_text,
    log_text=log_text,
)

print("Agent execution complete")
print("Available result keys:", list(result.keys()))

FAISS index created with 2 vectors
Agent execution complete
Available result keys: ['new_incident', 'similar_incidents', 'relevant_logs', 'log_analysis', 'triage_note']


In [9]:
for i, item in enumerate(result["similar_incidents"], 1):
    print("=" * 90)
    print(f"[{i}] Incident ID: {item.get('incident_id', 'unknown')}")
    print(f"Service: {item.get('service', 'unknown')}")
    print(f"Severity: {item.get('severity', 'unknown')}")
    print(f"Date: {item.get('date', 'unknown')}")
    print(f"Section: {item.get('section', 'unknown')}")
    print(f"Similarity Score: {item.get('score', 0.0):.3f}")
    print("\nContent:")
    print(item.get("text", "")[:900])

[1] Incident ID: INC-2026-001
Service: payment-service
Severity: Critical
Date: 2024-03-15
Section: summary
Similarity Score: 0.646

Content:
Payment service experienced high CPU utilization and increased checkout latency.
[2] Incident ID: INC-2026-001
Service: payment-service
Severity: Critical
Date: 2024-03-15
Section: root_cause
Similarity Score: 0.573

Content:
A recent deployment introduced inefficient database polling logic, causing CPU usage to spike above 90%.
[3] Incident ID: INC-2024-002
Service: order-service
Severity: High
Date: 2024-04-02
Section: summary
Similarity Score: 0.556

Content:
Order service experienced intermittent failures during checkout.
[4] Incident ID: INC-2026-001
Service: payment-service
Severity: Critical
Date: 2024-03-15
Section: prevention
Similarity Score: 0.517

Content:
Added CPU-based alerts, improved deployment validation, and introduced load testing for database polling changes.
[5] Incident ID: INC-2026-001
Service: payment-service
Severity: Cr

In [10]:
for i, item in enumerate(result["relevant_logs"], 1):
    print("=" * 90)
    print(f"[{i}] Timestamp: {item.get('timestamp', 'unknown')}")
    print(f"Level: {item.get('level', 'unknown')}")
    print(f"Service: {item.get('service', 'unknown')}")
    print(f"Similarity Score: {item.get('score', 0.0):.3f}")
    print(f"Error Hints: {item.get('error_hints', [])}")
    print("\nContent:")
    print(item.get("text", ""))

[1] Timestamp: 2026-04-26 10:21:11
Level: INFO
Service: payments-api
Similarity Score: 0.654
Error Hints: ['exception', 'timeout', 'timed out', 'connection pool exhausted', 'too many connections', 'failed', '500']

Content:
2026-04-26 10:21:11 INFO service=payments-api Starting request handling
2026-04-26 10:21:12 ERROR service=payments-api Connection pool exhausted for datasource main
2026-04-26 10:21:12 ERROR service=payments-api Failed to acquire DB connection after 30000ms
2026-04-26 10:21:13 WARN service=payments-api Request timed out for /checkout
2026-04-26 10:21:14 ERROR service=payments-api SQLException: too many connections
2026-04-26 10:21:15 ERROR service=payments-api Request failed with status 500
2026-04-26 10:21:18 INFO service=payments-api Deployment version 2.8.4 activated
2026-04-26 10:21:20 WARN service=payments-api Health check latency above threshold
2026-04-26 10:21:22 ERROR service=payments-api Timeout while calling downstream order service
2026-04-26 10:21:24 ER

In [11]:
print(result["log_analysis"])

1. What the logs suggest:
- After deployment of version 2.8.4 at 10:21:18, the payments-api experienced connection pool exhaustion errors.
- Multiple errors indicate failure to acquire database connections, resulting in SQLExceptions for "too many connections."
- Requests to /checkout are timing out and failing with HTTP 500 errors.
- There are also timeouts calling a downstream order service.
- Health check latency is above threshold, indicating degraded service responsiveness.

2. Likely technical failure pattern:
- The deployment introduced a regression causing the payments-api to exhaust its database connection pool.
- This leads to requests timing out waiting for DB connections, causing 500 errors and increased latency.
- The downstream order service calls also time out, possibly due to cascading delays or resource contention.
- Overall, the system is overwhelmed by too many concurrent DB connections or connections not being released properly.

3. Important unknowns / missing evid

In [12]:
print(result["triage_note"])

1. Likely Root Cause  
The recent deployment (version 2.8.4) introduced a regression causing the payments-api service to exhaust its database connection pool. This exhaustion leads to failed attempts to acquire DB connections, resulting in SQLExceptions ("too many connections"), request timeouts, and HTTP 500 errors during checkout. The downstream order service calls are also timing out, likely due to cascading delays or resource contention caused by the payments-api degradation.

2. Evidence from Logs  
- Deployment of version 2.8.4 activated at 10:21:18, immediately followed by:  
  - Errors indicating "Connection pool exhausted for datasource main" and "Failed to acquire DB connection after 30000ms."  
  - SQLException errors citing "too many connections."  
  - Request timeouts on /checkout endpoints and HTTP 500 failures.  
  - Timeouts calling the downstream order service.  
  - Health check latency above threshold, indicating degraded responsiveness.  
- No direct evidence of DB